# 04-2 실습 — 파일 조작과 작업 범위

전용 임시 디렉터리 안에서 새 파일을 만들고 복사·이름 변경·삭제합니다. 각 작업은 원본, 목적지, 충돌, 대상 종류를 먼저 확인합니다.

## Goal

- Path 생성, 디렉터리 생성, 파일 생성을 구분합니다.
- x 모드, shutil.copy2, Path.rename, Path.unlink, Path.rmdir를 사용합니다.
- 원본과 목적지가 같거나 목적지가 이미 존재하는 경우를 거부합니다.
- 삭제 전에 일반 파일과 심볼릭 링크를 구분합니다.
- 작업 범위와 실제 파일 작업의 예외를 함께 검증합니다.

## Setup

모든 변경은 TemporaryDirectory 내부에서만 일어납니다. 이 노트북의 삭제 함수는 재귀 삭제를 사용하지 않으며, 마지막의 전체 실습 정리는 임시 디렉터리 관리자가 담당합니다.

In [ ]:
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory

_lab_context = TemporaryDirectory(prefix='chapter04-operations-')
LAB_ROOT = Path(_lab_context.name).resolve(strict=True)
INPUT_DIR = LAB_ROOT / 'input'
OUTPUT_DIR = LAB_ROOT / 'output'
INPUT_DIR.mkdir()
OUTPUT_DIR.mkdir()

def expect_raises(expected_exception, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_exception as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f'{expected_exception.__name__} 대신 {type(exc).__name__} 발생'
        ) from exc
    raise AssertionError(f'{expected_exception.__name__}이 발생하지 않았습니다.')

print('격리된 실습 루트:', LAB_ROOT)

## Steps

### 1. 공통 경로 계약과 배타적 파일 생성

04-1의 resolve_under 계약을 그대로 사용합니다. 새 파일은 x 모드로 만들며, 같은 이름이 있으면 기존 내용을 보존한 채 FileExistsError를 발생시킵니다.

In [ ]:
def resolve_under(base, user_value, *, must_exist=False):
    if not isinstance(user_value, str):
        raise TypeError('경로 입력은 문자열이어야 합니다.')
    if not user_value.strip():
        raise ValueError('경로가 비어 있습니다.')

    raw_path = Path(user_value)
    if raw_path.is_absolute():
        raise ValueError('절대 경로는 허용하지 않습니다.')

    resolved_base = Path(base).resolve(strict=True)
    candidate = (resolved_base / raw_path).resolve(strict=must_exist)
    if not candidate.is_relative_to(resolved_base):
        raise ValueError('허용된 디렉터리 밖의 경로입니다.')
    return candidate

def create_new_text(base, user_value, text):
    if not isinstance(text, str):
        raise TypeError('파일 내용은 문자열이어야 합니다.')
    path = resolve_under(base, user_value, must_exist=False)
    if not path.parent.is_dir():
        raise NotADirectoryError(path.parent)
    with path.open('x', encoding='utf-8', newline='\n') as file:
        file.write(text)
    return path

source_path = create_new_text(LAB_ROOT, 'input/note.txt', 'practice\n')
assert source_path.read_text(encoding='utf-8') == 'practice\n'
expect_raises(FileExistsError, create_new_text, LAB_ROOT, 'input/note.txt', 'changed\n')
assert source_path.read_text(encoding='utf-8') == 'practice\n'

### 2. 덮어쓰지 않고 일반 파일 복사

사전 검사는 설명 가능한 오류를 만들기 위한 것이며, 실제 copy2 호출에서도 파일시스템 오류가 생길 수 있습니다. 메타데이터의 완전한 동일성은 운영체제마다 달라질 수 있으므로 내용과 대상 경로만 검증합니다.

In [ ]:
def copy_regular_file(base, source_value, destination_value):
    source = resolve_under(base, source_value, must_exist=True)
    destination = resolve_under(base, destination_value, must_exist=False)

    if not source.is_file():
        raise ValueError('원본이 일반 파일이 아닙니다.')
    if source == destination:
        raise ValueError('원본과 목적지가 같습니다.')
    if destination.exists():
        raise FileExistsError(destination)
    if not destination.parent.is_dir():
        raise NotADirectoryError(destination.parent)

    shutil.copy2(source, destination)
    return destination

copy_value = 'output/note-copy.txt'  # 'output/note-backup.txt'로 바꿔 보세요.
assert copy_value in {'output/note-copy.txt', 'output/note-backup.txt'}
copied_path = copy_regular_file(LAB_ROOT, 'input/note.txt', copy_value)
assert copied_path.read_text(encoding='utf-8') == 'practice\n'
assert source_path.read_text(encoding='utf-8') == 'practice\n'
print('복사 결과:', copied_path.relative_to(LAB_ROOT))

### 3. 충돌 없이 이름 변경

rename의 기존 목적지 처리 방식은 환경에 따라 다를 수 있으므로 목적지가 있으면 먼저 거부한다는 정책을 적용합니다.

In [ ]:
def rename_without_overwrite(base, before_value, after_value):
    before = resolve_under(base, before_value, must_exist=True)
    after = resolve_under(base, after_value, must_exist=False)

    if not before.is_file():
        raise ValueError('이동 원본이 일반 파일이 아닙니다.')
    if before == after:
        raise ValueError('이동 전후 경로가 같습니다.')
    if after.exists():
        raise FileExistsError(after)
    if not after.parent.is_dir():
        raise NotADirectoryError(after.parent)

    before.rename(after)
    return after

final_value = 'output/note-final.txt'
renamed_path = rename_without_overwrite(
    LAB_ROOT,
    copied_path.relative_to(LAB_ROOT).as_posix(),
    final_value,
)
assert renamed_path.is_file()
assert not copied_path.exists()
assert renamed_path.read_text(encoding='utf-8') == 'practice\n'

### 4. 결과 파일 한 개만 삭제

삭제 대상의 원래 디렉터리 항목이 심볼릭 링크인지 먼저 확인합니다. 이 함수는 일반 파일 한 개만 unlink하며 디렉터리나 원본 입력은 삭제하지 않습니다.

In [ ]:
def delete_regular_file(base, target_value):
    root = Path(base).resolve(strict=True)
    target = resolve_under(root, target_value, must_exist=True)
    original_entry = root / Path(target_value)

    if original_entry.is_symlink():
        raise ValueError('이 실습에서는 심볼릭 링크를 삭제하지 않습니다.')
    if not target.is_file():
        raise ValueError('삭제 대상이 일반 파일이 아닙니다.')

    target.unlink()

delete_regular_file(LAB_ROOT, final_value)
assert not renamed_path.exists()
assert source_path.is_file()
assert source_path.read_text(encoding='utf-8') == 'practice\n'

## Checks

없는 원본, 잘못된 대상 종류, 동일 경로, 기존 목적지, 작업 범위 이탈과 잘못된 부모를 점검합니다. 권한 오류는 실행 계정에 따라 재현되지 않을 수 있으므로 이 노트북에서 강제로 만들지 않습니다.

In [ ]:
existing_path = create_new_text(LAB_ROOT, 'output/existing.txt', 'keep\n')

expect_raises(FileNotFoundError, copy_regular_file, LAB_ROOT, 'input/missing.txt', 'output/new.txt')
expect_raises(ValueError, copy_regular_file, LAB_ROOT, 'input', 'output/input-copy')
expect_raises(ValueError, copy_regular_file, LAB_ROOT, 'input/note.txt', 'input/note.txt')
expect_raises(FileExistsError, copy_regular_file, LAB_ROOT, 'input/note.txt', 'output/existing.txt')
expect_raises(ValueError, copy_regular_file, LAB_ROOT, 'input/note.txt', '../outside.txt')
expect_raises(NotADirectoryError, copy_regular_file, LAB_ROOT, 'input/note.txt', 'missing/new.txt')

before_path = create_new_text(LAB_ROOT, 'output/before.txt', 'before\n')
after_path = create_new_text(LAB_ROOT, 'output/after.txt', 'after\n')
expect_raises(
    FileExistsError,
    rename_without_overwrite,
    LAB_ROOT,
    'output/before.txt',
    'output/after.txt',
)
assert before_path.is_file() and after_path.is_file()

empty_directory = LAB_ROOT / 'empty'
empty_directory.mkdir()
empty_directory.rmdir()
assert not empty_directory.exists()

nonempty_directory = LAB_ROOT / 'nonempty'
nonempty_directory.mkdir()
(nonempty_directory / 'child.txt').write_text('child\n', encoding='utf-8')
expect_raises(OSError, nonempty_directory.rmdir)
assert nonempty_directory.is_dir()

link_path = OUTPUT_DIR / 'note-link.txt'
try:
    link_path.symlink_to(source_path)
except (NotImplementedError, OSError):
    symlink_supported = False
else:
    symlink_supported = True
    expect_raises(ValueError, delete_regular_file, LAB_ROOT, 'output/note-link.txt')
    assert link_path.is_symlink()
    link_path.unlink()

assert existing_path.read_text(encoding='utf-8') == 'keep\n'
assert source_path.read_text(encoding='utf-8') == 'practice\n'
print('심볼릭 링크 점검 가능:', symlink_supported)
print('04-2 자기점검을 모두 통과했습니다.')

In [ ]:
_lab_context.cleanup()
assert not LAB_ROOT.exists()

## Next Steps

04-3에서는 파일 모드와 텍스트 줄바꿈 계약을 더 자세히 다룹니다. 실제 서비스에서는 사전 exists 검사와 작업 사이에 상태가 바뀔 수 있으므로 실제 copy, rename, unlink에서 발생한 예외도 프로그램 경계에서 처리해야 합니다.